# 01. Data preprocessing

This notebook prepares the baseline clinical dataframe for the four binary classification tasks used in the study.

The notebook performs:

- loading of the clinical dataframe
- task-specific label construction
- shared stratified train/test split
- feature column selection
- preprocessing utility definition
- task-wise sample count summary

In [2]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
GLOBAL_SEED = 17

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

set_seed(GLOBAL_SEED)

In [4]:
RAW_CSV = "../data/raw/adni_clinical_dataframe.csv"

OUTER_TEST_SIZE = 0.10
INNER_VAL_SIZE = 0.10

STRATIFY_COL = "DIAGNOSIS2"

In [5]:
DROP_COLS_CANDIDATES = [
    "PHASE", "PTID", "RID", "VISCODE2",
    "DIAGNOSIS", "DIAGNOSIS2", "CDGLOBAL"
]

TASKS = [
    {
        "name": "AD_vs_CN",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["AD"],
        "neg_labels": ["CN"],
    },
    {
        "name": "AD_vs_MCI",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["AD"],
        "neg_labels": ["MCI"],
    },
    {
        "name": "MCI_vs_CN",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["MCI"],
        "neg_labels": ["CN"],
    },
    {
        "name": "pMCI_vs_sMCI",
        "label_col": "DIAGNOSIS2",
        "pos_labels": ["pMCI"],
        "neg_labels": ["sMCI"],
    },
]

In [6]:
def assert_binary(y, where=""):
    classes = np.unique(y)
    if len(classes) < 2:
        raise ValueError(f"{where}: only one class present: {classes}")

def map_labels(df: pd.DataFrame, label_col: str, pos_labels, neg_labels):
    labels = df[label_col].astype(str).str.strip()

    pos_set = {str(x).strip() for x in pos_labels}
    neg_set = {str(x).strip() for x in neg_labels}

    mask = labels.isin(pos_set.union(neg_set))
    sub_df = df.loc[mask].copy().reset_index(drop=True)

    y = sub_df[label_col].astype(str).str.strip().isin(pos_set).astype(int).values

    return sub_df, y

In [7]:
def is_cat_col(series: pd.Series) -> bool:
    dtype = series.dtype

    if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_bool_dtype(dtype):
        return True

    return isinstance(dtype, pd.CategoricalDtype)

def build_preprocessor(train_df: pd.DataFrame, feature_cols):
    cat_cols, num_cols = [], []

    for col in feature_cols:
        if is_cat_col(train_df[col]):
            cat_cols.append(col)
        else:
            num_cols.append(col)

    cat_maps = {}

    for col in cat_cols:
        values = train_df[col].astype("object")
        values = values.where(~values.isna(), "__MISSING__").astype(str)
        unique_values = pd.unique(values)

        cat_maps[col] = {val: i + 1 for i, val in enumerate(unique_values)}

    return cat_cols, num_cols, cat_maps

def transform_df(df: pd.DataFrame, feature_cols, cat_cols, num_cols, cat_maps):
    X = np.zeros((len(df), len(feature_cols)), dtype=np.float32)

    for j, col in enumerate(feature_cols):
        if col in cat_cols:
            values = df[col].astype("object")
            values = values.where(~values.isna(), "__MISSING__").astype(str)
            X[:, j] = np.array([cat_maps[col].get(v, 0) for v in values], dtype=np.float32)
        else:
            values = pd.to_numeric(df[col], errors="coerce").astype(np.float32)
            X[:, j] = np.nan_to_num(values.values, nan=0.0)

    return X

In [8]:
df_all = pd.read_csv(RAW_CSV).copy()

df_all[STRATIFY_COL] = df_all[STRATIFY_COL].astype(str).str.strip()
df_all = df_all[df_all[STRATIFY_COL].notna()].copy()
df_all = df_all[df_all[STRATIFY_COL] != ""].copy()

print("Data shape:", df_all.shape)
df_all.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/adni_clinical_dataframe.csv'

In [ ]:
df_train_all, df_test_all = train_test_split(
    df_all,
    test_size=OUTER_TEST_SIZE,
    random_state=GLOBAL_SEED,
    stratify=df_all[STRATIFY_COL].values,
)

df_train_all = df_train_all.reset_index(drop=True)
df_test_all = df_test_all.reset_index(drop=True)

print("Train samples:", len(df_train_all))
print("Test samples:", len(df_test_all))

print("\nTrain distribution:")
print(df_train_all[STRATIFY_COL].value_counts())

print("\nTest distribution:")
print(df_test_all[STRATIFY_COL].value_counts())

In [ ]:
PROCESSED_DIR = "../data/processed"
SPLIT_DIR = "../data/splits"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs("../results", exist_ok=True)

task_summaries = []

for task in TASKS:
    task_name = task["name"]

    # Select task-specific samples from the shared outer split
    df_train_task, y_train = map_labels(
        df_train_all,
        task["label_col"],
        task["pos_labels"],
        task["neg_labels"],
    )

    df_test_task, y_test = map_labels(
        df_test_all,
        task["label_col"],
        task["pos_labels"],
        task["neg_labels"],
    )

    assert_binary(y_train, f"{task_name} train")
    assert_binary(y_test, f"{task_name} test")

    # Select feature columns
    drop_cols = set(DROP_COLS_CANDIDATES + [task["label_col"]])
    feature_cols = [col for col in df_train_task.columns if col not in drop_cols]

    # Fit categorical mapping on training data only
    cat_cols, num_cols, cat_maps = build_preprocessor(df_train_task, feature_cols)

    # Transform train and test using training-derived mappings
    X_train = transform_df(df_train_task, feature_cols, cat_cols, num_cols, cat_maps)
    X_test = transform_df(df_test_task, feature_cols, cat_cols, num_cols, cat_maps)

    # Save processed train/test files
    train_out = pd.DataFrame(X_train, columns=feature_cols)
    train_out.insert(0, "label", y_train)

    test_out = pd.DataFrame(X_test, columns=feature_cols)
    test_out.insert(0, "label", y_test)

    train_path = f"{PROCESSED_DIR}/{task_name}_train.csv"
    test_path = f"{PROCESSED_DIR}/{task_name}_test.csv"

    train_out.to_csv(train_path, index=False)
    test_out.to_csv(test_path, index=False)

    # Save feature list for each task
    feature_path = f"{SPLIT_DIR}/{task_name}_features.txt"
    with open(feature_path, "w", encoding="utf-8") as f:
        for col in feature_cols:
            f.write(col + "\n")

    task_summaries.append({
        "task": task_name,
        "positive_class": task["pos_labels"][0],
        "negative_class": task["neg_labels"][0],
        "n_features": len(feature_cols),
        "train_n": len(y_train),
        "test_n": len(y_test),
        "positive_train_n": int((y_train == 1).sum()),
        "negative_train_n": int((y_train == 0).sum()),
        "positive_test_n": int((y_test == 1).sum()),
        "negative_test_n": int((y_test == 0).sum()),
        "train_file": train_path,
        "test_file": test_path,
        "feature_file": feature_path,
    })

task_summary_df = pd.DataFrame(task_summaries)
task_summary_df

In [ ]:
os.makedirs("../results", exist_ok=True)

split_summary.to_csv("../results/task_split_summary.csv", index=False)
print("Saved: ../results/task_split_summary.csv")